In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import *


### Data Reading


In [0]:
df = spark.read.format("parquet")\
    .load("abfss://bronze@adbstoragepro.dfs.core.windows.net/orders")


In [0]:
display(df)
df.printSchema()

In [0]:
df.withColumnRenamed("_rescued_data","rescued_data")
display(df)
df= df.drop("rescude_data")
display(df)

In [0]:
df = df.withColumn("order_date",to_timestamp(col("order_date")))
display(df)

In [0]:
df =df.withColumn("year", year(col("order_date")))
display(df)

In [0]:
from pyspark.sql.window import Window
df1 = df.withColumn("flag",dense_rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
display(df1)

In [0]:
df1 = df1.withColumn("rank_flag",rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
display(df1)

In [0]:
df1 = df1.withColumn("row_flag",row_number().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
display(df1)

### Classes -OOP

In [0]:
class windows:
    def dense_rank(self,df):
        df_dense_rank =df.withColumn("flag_dense_rank",dense_rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
        return df_dense_rank
    def rank(self, df):
         df_rank =df.withColumn("flag_rank",rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
         return df_rank
    def row_number(self, df):
         df_row_number =df.withColumn("flag_row_number",row_number().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
         return df_row_number


In [0]:
df_new = df
display(df_new)

In [0]:
obj = windows()

In [0]:
#reusing classes
df_result=obj.dense_rank(df_new)
display(df_result)

### Data Writing

In [0]:
df.write.format("delta").mode("overwrite").save("abfss://silver@adbstoragepro.dfs.core.windows.net/orders")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS DEV.SILVER.orders_silver
    USING DELTA
    LOCATION 'abfss://silver@adbstoragepro.dfs.core.windows.net/orders';